# Advanced Model Training
Here we load the featured datasets, remove outliers, handle grouped imputation, and run an advanced scikit-learn pipeline using XGBoost, Target Encoding, and Model Stacking!

In [32]:
import pandas as pd
import numpy as np
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder, TargetEncoder
from sklearn.feature_selection import SelectFromModel
from sklearn.linear_model import RidgeCV, LinearRegression, LassoCV
from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.ensemble import StackingRegressor
from xgboost import XGBRegressor

In [33]:
# 1. Load data
train_df = pd.read_csv('Ames Housing/datasets/train_featured.csv')
test_df = pd.read_csv('Ames Housing/datasets/test_featured.csv')

# 2. Separate X and y
X_train = train_df.drop('SalePrice', axis=1)
y_train = train_df['SalePrice']

X_test = test_df.drop('SalePrice', axis=1)
y_test = np.log1p(test_df['SalePrice']) # Log transform test target for equivalent evaluation

In [34]:
# 3. Outlier removal on TRAIN only
outlier_mask = (X_train["Gr Liv Area"] > 4000) & (y_train < np.log1p(300000))

X_train = X_train[~outlier_mask]
y_train = y_train[~outlier_mask]
print(f"Removed {outlier_mask.sum()} outliers from training data.")

Removed 2 outliers from training data.


In [35]:
# 4. Custom Grouped Median Imputer
class GroupedMedianImputer(BaseEstimator, TransformerMixin):
    def __init__(self, group_col, target_col):
        self.group_col = group_col
        self.target_col = target_col
        
    def fit(self, X, y=None):
        self.medians_ = X.groupby(self.group_col)[self.target_col].median().to_dict()
        self.global_median_ = X[self.target_col].median()
        return self
        
    def transform(self, X):
        X_copy = X.copy()
        X_copy[self.target_col] = X_copy.apply(
            lambda row: self.medians_.get(row[self.group_col], self.global_median_) 
            if pd.isna(row[self.target_col]) else row[self.target_col], axis=1
        )
        return X_copy

grouped_imputer = GroupedMedianImputer(group_col='Neighborhood', target_col='Lot Frontage')
X_train = grouped_imputer.fit_transform(X_train)
X_test = grouped_imputer.transform(X_test)

### The Advanced Pipeline Setup
Instead of just One-Hot Encoding, we use **Target Encoding** for high-cardinality features like Neighborhoods. This prevents the dataset from expanding to hundreds of columns and gives tree-based models a strong continuous signal.

In [36]:
# 5. Build Scikit-Learn Pipeline
num_cols = X_train.select_dtypes(include=['int64', 'float64']).columns.tolist()
cat_cols = X_train.select_dtypes(include=['object']).columns.tolist()

numeric_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler())
])

# ADVANCED UPGRADE: Using TargetEncoder for categorical columns!
categorical_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('encoder', TargetEncoder(target_type='continuous')) 
])

preprocessor = ColumnTransformer(
    transformers=[
        ('num', numeric_transformer, num_cols),
        ('cat', categorical_transformer, cat_cols)
    ], remainder='passthrough')

### Model 1: Baseline Ridge Regression

In [37]:
# Train standard Ridge with Lasso Feature Selection to set a baseline
ridge_model = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('feature_selection', SelectFromModel(LassoCV(cv=5), max_features=52, threshold=-np.inf)),
    ('regressor', RidgeCV(alphas=(0.1, 1.0, 10.0, 100.0)))
])

ridge_model.fit(X_train, y_train)
y_pred_ridge = ridge_model.predict(X_test)

print(f"--- Ridge Results (With Target Encoding) ---")
print(f"RMSE (Log Scale): {np.sqrt(mean_squared_error(y_test, y_pred_ridge)):.4f}")
print(f"R2 Score: {r2_score(y_test, y_pred_ridge):.4f}")

--- Ridge Results (With Target Encoding) ---
RMSE (Log Scale): 0.1268
R2 Score: 0.9131


### Model 2: XGBoost Regressor
XGBoost handles complex non-linear relationships better than Ridge.

In [38]:
# Train XGBoost with Lasso Feature Selection
xgb_model = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('feature_selection', SelectFromModel(LassoCV(cv=5), max_features=52, threshold=-np.inf)),
    ('regressor', XGBRegressor(n_estimators=500, learning_rate=0.05, max_depth=4, random_state=42))
])

xgb_model.fit(X_train, y_train)
y_pred_xgb = xgb_model.predict(X_test)

print(f"--- XGBoost Results ---")
print(f"RMSE (Log Scale): {np.sqrt(mean_squared_error(y_test, y_pred_xgb)):.4f}")
print(f"R2 Score: {r2_score(y_test, y_pred_xgb):.4f}")

--- XGBoost Results ---
RMSE (Log Scale): 0.1087
R2 Score: 0.9361


### Model 3: Stacking Regressor (The Ultimate Model)
We combine the predictions of Ridge and XGBoost. A final Linear Regression model learns when to trust Ridge vs when to trust XGBoost.

In [39]:
# Train Stacking Regressor
estimators = [
    ('ridge', RidgeCV(alphas=(0.1, 1.0, 10.0, 100.0))),
    ('xgb', XGBRegressor(n_estimators=300, learning_rate=0.05, max_depth=4, random_state=42))
]

stacked_model = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('feature_selection', SelectFromModel(LassoCV(cv=5), max_features=52, threshold=-np.inf)),
    ('regressor', StackingRegressor(
        estimators=estimators,
        final_estimator=LinearRegression()
    ))
])

stacked_model.fit(X_train, y_train)
y_pred_stacked = stacked_model.predict(X_test)

print(f"--- Stacking Regressor Results ---")
print(f"RMSE (Log Scale): {np.sqrt(mean_squared_error(y_test, y_pred_stacked)):.4f}")
print(f"R2 Score: {r2_score(y_test, y_pred_stacked):.4f}")

--- Stacking Regressor Results ---
RMSE (Log Scale): 0.1147
R2 Score: 0.9289
